In [ ]:
# Load the required packages
import scanorama
import scanpy as sc
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time
import tracemalloc
import os
import psutil
import anndata as ad
import numpy as np
from scipy import sparse

In [ ]:
# Set up the memory tracker
process = psutil.Process(os.getpid())

def rss_mb():
    return process.memory_info().rss / 1024 / 1024

In [ ]:
# Load data 
print("RSS before:", rss_mb(), "MB")
adata = sc.read_h5ad("./prep_FINAL.h5ad")
print("RSS after:", rss_mb(), "MB")

In [ ]:
# Converting the expression matrix to the Compressed Sparse Row format for more efficient processing
adata.X = adata.X.tocsr()

In [ ]:
# Normalize the counts:
sc.pp.normalize_per_cell(adata, counts_per_cell_after=1e4)
sc.pp.log1p(adata)

## Split the object by subject/individua

In [ ]:
# 1. Split the anndata object into a disctionary of objects based on the individual id
split_column = 'new_id'
categories = adata.obs[split_column].unique()

# Create a dictionary to hold the split AnnData objects
split_adata_dict = {
    category: adata[adata.obs[split_column] == category, :].copy()
    for category in categories
}

print(f"Split AnnData into {len(split_adata_dict)} objects.")

## Batch correction and integration with Scanorama

In [ ]:
# --- 2. Loop through and perform operations ---

start = time.perf_counter() # track time
tracemalloc.start() # track memory
print("RSS before:", rss_mb(), "MB")

corrected_dict = {}

# Loop through each category and its corresponding AnnData object
for category, subset_adata in split_adata_dict.items():
    print(f"\nProcessing subset for category: '{category}'")

    # From tutorial:
    batch_key = 'assay'
    split_adata_assay = [subset_adata[subset_adata.obs[batch_key] == batch_value].copy() for batch_value in subset_adata.obs[batch_key].unique()] 
    print(split_adata_assay)

    # Scanorama batch correction
    # Creates new AnnData objects and replace adata.X with the Scanorama-transformed cell-by-gene matrix, 
    # while keeping the other metadata in adata as well.
    print("Running scanorama")
    corrected = scanorama.correct_scanpy(split_adata_assay, return_dimred=True, approx=False) 
    

    end = time.perf_counter()
    print(f"Total loop runtime: {end - start:.4f} seconds")
    current, peak = tracemalloc.get_traced_memory()
    print(f"Peak memory during loop: {peak / 1024 / 1024:.2f} MB")
    print("RSS after:", rss_mb(), "MB")

    # Concatenate the assays back together
    merged_corrected = ad.concat(
        corrected,
        join='outer',
        label='original_batch',
        fill_value=0
    )
    
    # Store the modified AnnData object in a new dictionary
    corrected_dict[category] = merged_corrected

print(f"Peak memory after loop: {peak / 1024 / 1024:.2f} MB")
tracemalloc.stop()

In [ ]:
# --- 3. Concatenate the modified objects back together ---
merged_adata = ad.concat(
    corrected_dict,
    join='outer',
    label='original_batch',
    fill_value=0
)

In [ ]:
# Save the dataset
merged_adata.write("./DS_scanorama.h5ad") # Specify the dataset

In [ ]:
## OR for the UseCase
merged_adata.write("./DS_merged_scanorama_25.h5ad")